# Multi-Temporal Land Use & Land Cover (LULC) Classification & Semantic Change Detection
**Model Architecture:** Siamese Swin-Tiny + Bottleneck Self-Attention (BSA) + ASPP + Cross-Transformer  
**Dataset:** Sentinel-2 Multi-Spectral Surface Reflectance (2017, 2019, 2024, 2025) — Chennai Region  
**Supervision:** 8-Class ESA WorldCover Ground Truth + Water-Gated Change Vector Analysis (CVA)

## 1. Environment Setup & Hyperparameter Configuration

In [1]:
from huggingface_hub import login
try:
    hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
except Exception:
    # login via HF_TOKEN environment variable if available
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)

In [2]:


import csv
import os
import random
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_NAME = "training_final_5"
def resolve_path(candidates, default):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return default

DATA_DIR = resolve_path([os.getenv("DATA_DIR", ""), "./data", "./dataset", "/kaggle/input/lulc-chennai", "../data"], "./data")
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)

NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"

RESUME_CHECKPOINT = None

CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60
SENSITIVITY_TAUS = (0.40, 0.50, 0.60, 0.70, 0.80)
SENSITIVITY_PATCHES_PER_PAIR = 20

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Run: {RUN_NAME} | Device: {device} | Checkpoint: {CKPT_NAME}")

Run: training_final_5 | Device: cuda | Checkpoint: best_model_final_5.pth


## 2. Multi-Temporal Dataset Loading & Spatial Disjoint Split

In [3]:
patches_2017 = np.load(os.path.join(DATA_DIR, "patches_2017.npy"))
patches_2019 = np.load(os.path.join(DATA_DIR, "patches_2019.npy"))
patches_2024 = np.load(os.path.join(DATA_DIR, "patches_2024.npy"))
patches_2025 = np.load(os.path.join(DATA_DIR, "patches_2025.npy"))
labels = np.load(os.path.join(DATA_DIR, "label_patches.npy"))

YEAR_ARRAYS = {
    2017: patches_2017,
    2019: patches_2019,
    2024: patches_2024,
    2025: patches_2025,
}
LABEL_FOR_YEAR = {year: labels for year in YEAR_ARRAYS}

n_patches = labels.shape[0]
rng = np.random.RandomState(SEED)
shuffled_idx = rng.permutation(n_patches)
n_train = int(n_patches * 0.80)
train_patch_idx = shuffled_idx[:n_train]
val_patch_idx = shuffled_idx[n_train:]
assert set(train_patch_idx).isdisjoint(set(val_patch_idx))
print(f"Disjoint patch split: {len(train_patch_idx)} train / {len(val_patch_idx)} validation")


Disjoint patch split: 2774 train / 694 validation


## 3. Spectral Feature Engineering & Water-Gated CVA Change Detection

In [4]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out


def build_nine_band_array(array):
    result = np.zeros((array.shape[0], 9, array.shape[2], array.shape[3]), dtype=np.float32)
    for index in range(array.shape[0]):
        result[index] = normalize_bands(to_nine_band(array[index]))
    return result


nine_band = {year: build_nine_band_array(array) for year, array in YEAR_ARRAYS.items()}


def cva_change_mask(img1, img2):
    diff = img1.astype(np.float64) - img2.astype(np.float64)
    magnitude = np.sqrt((diff ** 2).sum(axis=0))
    try:
        threshold = threshold_otsu(magnitude)
    except ValueError:
        threshold = 0.5
    changed = (magnitude > threshold).astype(np.float32)
    stable_water = (img1[6] > CVA_WATER_NDWI_THRESHOLD) & (img2[6] > CVA_WATER_NDWI_THRESHOLD)
    changed[stable_water] = 0.0
    return changed


YEAR_PAIRS = [
    (2017, 2019), (2017, 2024), (2017, 2025),
    (2019, 2024), (2019, 2025), (2024, 2025),
]

cva_masks = {}
for year1, year2 in YEAR_PAIRS:
    for patch_index in range(n_patches):
        cva_masks[(year1, year2, patch_index)] = cva_change_mask(
            nine_band[year1][patch_index], nine_band[year2][patch_index]
        )
print(f"Precomputed water-gated CVA masks for {len(YEAR_PAIRS)} temporal pairs.")

del nine_band
import gc
gc.collect()
print("Cleaned up precomputed nine_band array from memory.")

Precomputed water-gated CVA masks for 6 temporal pairs.
Cleaned up precomputed nine_band array from memory.


## 4. Siamese Dataset & Synchronized Augmentations

In [5]:
class PairIndex:
    def __init__(self, pairs, patch_indices):
        self.items = [(year1, year2, index) for year1, year2 in pairs for index in patch_indices]

    def __len__(self):
        return len(self.items)


train_index = PairIndex(YEAR_PAIRS, train_patch_idx)
val_index = PairIndex(YEAR_PAIRS, val_patch_idx)


class SiameseLULCDataset(Dataset):
    def __init__(self, pair_index, patches_by_year, labels_by_year, cva_by_pair, augment=False):
        self.items = pair_index.items
        self.patches = patches_by_year
        self.labels = labels_by_year
        self.cva = cva_by_pair
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, item_index):
        year1, year2, patch_index = self.items[item_index]
        img1 = self.patches[year1][patch_index].copy()
        img2 = self.patches[year2][patch_index].copy()
        label = self.labels[year2][patch_index].astype(np.int64)
        change = self.cva[(year1, year2, patch_index)].copy()

        if self.augment:
            rotations = random.randint(0, 3)
            if rotations:
                img1 = np.rot90(img1, rotations, (1, 2)).copy()
                img2 = np.rot90(img2, rotations, (1, 2)).copy()
                label = np.rot90(label, rotations, (0, 1)).copy()
                change = np.rot90(change, rotations, (0, 1)).copy()
            if random.random() < 0.5:
                img1, img2 = np.flip(img1, 2).copy(), np.flip(img2, 2).copy()
                label, change = np.flip(label, 1).copy(), np.flip(change, 1).copy()
            if random.random() < 0.5:
                img1, img2 = np.flip(img1, 1).copy(), np.flip(img2, 1).copy()
                label, change = np.flip(label, 0).copy(), np.flip(change, 0).copy()
            if random.random() < 0.3:
                alpha = 0.9 + random.random() * 0.2
                beta = -0.05 + random.random() * 0.1
                img1 = np.clip(img1 * alpha + beta, 0, 1)
                img2 = np.clip(img2 * alpha + beta, 0, 1)

        img1 = normalize_bands(to_nine_band(img1))
        img2 = normalize_bands(to_nine_band(img2))
        return (
            torch.from_numpy(img1),
            torch.from_numpy(img2),
            torch.from_numpy(label),
            torch.from_numpy(change),
        )


train_loader = DataLoader(
    SiameseLULCDataset(train_index, YEAR_ARRAYS, LABEL_FOR_YEAR, cva_masks, augment=True),
    batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"],
    pin_memory=True, drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    SiameseLULCDataset(val_index, YEAR_ARRAYS, LABEL_FOR_YEAR, cva_masks, augment=False),
    batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"],
    pin_memory=True, persistent_workers=True,
)


def estimate_change_positive_weight(pair_index, masks, cap):
    positives = 0.0
    total = 0
    for year1, year2, patch_index in pair_index.items:
        mask = masks[(year1, year2, patch_index)]
        positives += float(mask.sum())
        total += mask.size
    rate = positives / max(total, 1)
    weight = min((total - positives) / max(positives, 1.0), cap)
    return rate, float(max(weight, 1.0))


CHANGE_POSITIVE_RATE, CHANGE_POS_WEIGHT = estimate_change_positive_weight(
    train_index, cva_masks, CONFIG["change_pos_weight_cap"]
)
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")
print(f"Water-gated CVA positive rate: {CHANGE_POSITIVE_RATE:.3%} | BCE positive weight: {CHANGE_POS_WEIGHT:.3f}")

Train batches: 520 | Validation batches: 131
Water-gated CVA positive rate: 10.912% | BCE positive weight: 8.000


## 5. Model Architecture (Siamese Swin-Tiny + ASPP)

In [6]:
def compute_capped_class_weights(labels_arr, num_classes, cap):
    flat = labels_arr.reshape(-1)
    counts = np.bincount(flat, minlength=num_classes).astype(np.float64)
    counts[counts==0] = 1
    freq = counts/counts.sum(); inv = 1/freq; inv = inv/inv.mean()
    return torch.tensor(np.clip(inv,None,cap), dtype=torch.float32)

class_weights = compute_capped_class_weights(labels, NUM_CLASSES, CONFIG["class_weight_cap"]).to(device)

class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=True,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

print(f"Model: {CONFIG['backbone']} + ASPP ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Model: swin_tiny_patch4_window7_224 + ASPP (84.24M params)


## 6. Multi-Task Losses & Evaluation Metrics

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, class_weights, gamma=1.0, label_smoothing=0.0):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        log_prob = F.log_softmax(logits, dim=1)
        prob = log_prob.exp()
        classes = logits.shape[1]
        one_hot = F.one_hot(target, classes).permute(0, 3, 1, 2).float()
        if self.label_smoothing:
            one_hot = one_hot * (1 - self.label_smoothing) + self.label_smoothing / classes
        focal = -((1 - prob).clamp(min=0.0, max=1.0) ** self.gamma) * log_prob * one_hot
        weighted = focal * self.class_weights.view(1, -1, 1, 1)
        return weighted.sum(dim=1).mean()


def lovasz_grad(gt_sorted):
    total = gt_sorted.sum()
    intersection = total - gt_sorted.float().cumsum(0)
    union = total + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if gt_sorted.numel() > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard


def lovasz_softmax_flat(probabilities, labels, classes="present"):
    losses = []
    for class_index in range(probabilities.size(1)):
        foreground = (labels == class_index).float()
        if classes == "present" and foreground.sum() == 0:
            continue
        errors = (foreground - probabilities[:, class_index]).abs()
        errors_sorted, permutation = torch.sort(errors, descending=True)
        losses.append(torch.dot(errors_sorted, lovasz_grad(foreground[permutation])))
    return sum(losses) / max(len(losses), 1)


class LovaszSoftmax(nn.Module):
    def __init__(self, classes="present"):
        super().__init__()
        self.classes = classes

    def forward(self, logits, labels):
        probabilities = F.softmax(logits, dim=1).permute(0, 2, 3, 1).contiguous().view(-1, logits.size(1))
        return lovasz_softmax_flat(probabilities, labels.reshape(-1), self.classes)


class BalancedChangeLoss(nn.Module):
    def __init__(self, pos_weight, bce_weight=0.7, dice_weight=0.3, eps=1e-6):
        super().__init__()
        self.register_buffer("pos_weight", torch.tensor(float(pos_weight)))
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits, target):
        target = target.float()
        bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pos_weight)
        probability = torch.sigmoid(logits)
        intersection = (probability * target).sum(dim=(1, 2))
        denominator = probability.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
        dice_loss = 1 - ((2 * intersection + self.eps) / (denominator + self.eps)).mean()
        return self.bce_weight * bce + self.dice_weight * dice_loss


def build_criteria(focal_gamma):
    focal = FocalLoss(class_weights, focal_gamma, CONFIG["label_smoothing"])
    lovasz = LovaszSoftmax("present")
    change = BalancedChangeLoss(
        CHANGE_POS_WEIGHT,
        CONFIG["change_bce_weight"],
        CONFIG["change_dice_weight"],
    ).to(device)
    return focal, lovasz, change


def combined_loss(lulc_logits, change_logits, label, cva_target, focal, lovasz, change_loss):
    focal_loss = focal(lulc_logits, label)
    lovasz_loss = lovasz(lulc_logits, label)
    cva_change_loss = change_loss(change_logits, cva_target)
    total = focal_loss + lovasz_loss + CONFIG["change_task_weight"] * cva_change_loss
    return total, focal_loss, lovasz_loss, cva_change_loss


def compute_confusion(prediction, truth, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for true_class in range(num_classes):
        for predicted_class in range(num_classes):
            matrix[true_class, predicted_class] = np.sum(
                (truth == true_class) & (prediction == predicted_class)
            )
    return matrix


def metrics_from_confusion(matrix):
    overall_accuracy = np.trace(matrix) / matrix.sum()
    iou = np.zeros(matrix.shape[0])
    f1 = np.zeros(matrix.shape[0])
    for class_index in range(matrix.shape[0]):
        true_positive = matrix[class_index, class_index]
        false_positive = matrix[:, class_index].sum() - true_positive
        false_negative = matrix[class_index, :].sum() - true_positive
        iou[class_index] = true_positive / (true_positive + false_positive + false_negative + 1e-12)
        precision = true_positive / (true_positive + false_positive + 1e-12)
        recall = true_positive / (true_positive + false_negative + 1e-12)
        f1[class_index] = 2 * precision * recall / (precision + recall + 1e-12)
    return overall_accuracy, iou.mean(), f1.mean(), iou, f1


def binary_metrics(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    true_positive = np.logical_and(prediction, target).sum()
    false_positive = np.logical_and(prediction, ~target).sum()
    false_negative = np.logical_and(~prediction, target).sum()
    precision = true_positive / (true_positive + false_positive + 1e-12)
    recall = true_positive / (true_positive + false_negative + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    iou = true_positive / (true_positive + false_positive + false_negative + 1e-12)
    return {"precision": precision, "recall": recall, "f1": f1, "iou": iou}


def tta_predict(model, x1, x2):
    lulc_sum = None
    change_sum = None
    for rotation in range(4):
        rotated_x1 = torch.rot90(x1, rotation, (2, 3))
        rotated_x2 = torch.rot90(x2, rotation, (2, 3))
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(rotated_x1, rotated_x2)
        lulc_logits = torch.rot90(lulc_logits, -rotation, (2, 3))
        change_logits = torch.rot90(change_logits.unsqueeze(1), -rotation, (2, 3)).squeeze(1)
        lulc_sum = lulc_logits if lulc_sum is None else lulc_sum + lulc_logits
        change_sum = change_logits if change_sum is None else change_sum + change_logits
    return lulc_sum / 4, change_sum / 4

## 7. Model Training & Validation Engine

In [8]:
def set_backbone_trainable(model, trainable):
    for parameter in model.backbone.parameters():
        parameter.requires_grad = trainable


def discriminative_optimizer(model, learning_rate, weight_decay, backbone_ratio):
    backbone_params = [parameter for parameter in model.backbone.parameters() if parameter.requires_grad]
    backbone_ids = {id(parameter) for parameter in model.backbone.parameters()}
    head_params = [
        parameter for parameter in model.parameters()
        if id(parameter) not in backbone_ids and parameter.requires_grad
    ]
    parameter_groups = [{"params": head_params, "lr": learning_rate}]
    if backbone_params:
        parameter_groups.insert(0, {"params": backbone_params, "lr": learning_rate * backbone_ratio})
    return torch.optim.AdamW(parameter_groups, weight_decay=weight_decay)


def run_train(model, loader, optimizer, scaler, focal, lovasz, change_loss):
    model.train()
    total_loss = 0.0
    for x1, x2, label, cva in loader:
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        cva = cva.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(x1, x2)
            loss, _, _, _ = combined_loss(lulc_logits, change_logits, label, cva, focal, lovasz, change_loss)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


def run_eval(model, loader, use_tta=False, focal=None, lovasz=None, change_loss=None):
    model.eval()
    focal = focal or FocalLoss(class_weights, 1.0)
    lovasz = lovasz or LovaszSoftmax("present")
    change_loss = change_loss or BalancedChangeLoss(
        CHANGE_POS_WEIGHT, CONFIG["change_bce_weight"], CONFIG["change_dice_weight"]
    ).to(device)
    matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    change_predictions, change_targets, label_values, prediction_values = [], [], [], []
    total_loss = 0.0
    for x1, x2, label, cva in loader:
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        label_device = label.to(device, non_blocking=True)
        cva_device = cva.to(device, non_blocking=True)
        if use_tta:
            lulc_logits, change_logits = tta_predict(model, x1, x2)
        else:
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                lulc_logits, change_logits = model(x1, x2)
        loss, _, _, _ = combined_loss(
            lulc_logits, change_logits, label_device, cva_device, focal, lovasz, change_loss
        )
        total_loss += loss.item()
        prediction = lulc_logits.argmax(1).cpu().numpy()
        truth = label.numpy()
        matrix += compute_confusion(prediction.reshape(-1), truth.reshape(-1), NUM_CLASSES)
        change_predictions.append((torch.sigmoid(change_logits) > 0.5).cpu().numpy().reshape(-1))
        change_targets.append(cva.numpy().reshape(-1))
        label_values.append(truth.reshape(-1))
        prediction_values.append(prediction.reshape(-1))

    overall_accuracy, mean_iou, mean_f1, iou_per_class, f1_per_class = metrics_from_confusion(matrix)
    cva_metrics = binary_metrics(np.concatenate(change_predictions), np.concatenate(change_targets))
    labels_flat = np.concatenate(label_values)
    predictions_flat = np.concatenate(prediction_values)
    subset = np.random.choice(len(labels_flat), min(200000, len(labels_flat)), replace=False)
    kappa = cohen_kappa_score(labels_flat[subset], predictions_flat[subset])
    return {
        "loss": total_loss / len(loader),
        "oa": overall_accuracy,
        "miou": mean_iou,
        "mf1": mean_f1,
        "kappa": kappa,
        "cva_change_precision": cva_metrics["precision"],
        "cva_change_recall": cva_metrics["recall"],
        "cva_change_f1": cva_metrics["f1"],
        "cva_change_iou": cva_metrics["iou"],
        "iou_per_class": iou_per_class,
        "f1_per_class": f1_per_class,
    }


def train_full(model, train_loader, validation_loader, checkpoint_path):
    focal, lovasz, change_loss = build_criteria(CONFIG["focal_gamma"])
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    start_epoch = 1
    best_miou = -1.0

    set_backbone_trainable(model, False)
    optimizer = discriminative_optimizer(model, CONFIG["lr"], CONFIG["weight_decay"], CONFIG["backbone_lr_ratio"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["freeze_epochs"], eta_min=1e-6
    )

    if RESUME_CHECKPOINT:
        if not os.path.exists(RESUME_CHECKPOINT):
            raise FileNotFoundError(f"RESUME_CHECKPOINT does not exist: {RESUME_CHECKPOINT}")
        state = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        model.load_state_dict(state["model_state_dict"])
        start_epoch = state["epoch"] + 1
        best_miou = state.get("val_metrics", {}).get("miou", -1.0)
        print(f"Resuming V5 from epoch {start_epoch}; best mIoU={best_miou:.4f}")

    # Ensure correct backbone status and scheduler if resumed past the freeze stage
    if start_epoch > CONFIG["freeze_epochs"]:
        set_backbone_trainable(model, True)
        optimizer = discriminative_optimizer(
            model, CONFIG["lr"], CONFIG["weight_decay"], CONFIG["backbone_lr_ratio"]
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=CONFIG["epochs"] - CONFIG["freeze_epochs"],
            eta_min=1e-6,
        )
        for _ in range(start_epoch - CONFIG["freeze_epochs"] - 1):
            scheduler.step()
        print(f"Resumed beyond freeze stage: backbone unfrozen, scheduler advanced.")
    elif start_epoch > 1:
        for _ in range(start_epoch - 1):
            scheduler.step()

    patience = 0
    for epoch in range(start_epoch, CONFIG["epochs"] + 1):
        started = time.time()
        if epoch == CONFIG["freeze_epochs"] + 1 and start_epoch <= CONFIG["freeze_epochs"]:
            set_backbone_trainable(model, True)
            optimizer = discriminative_optimizer(
                model, CONFIG["lr"], CONFIG["weight_decay"], CONFIG["backbone_lr_ratio"]
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=CONFIG["epochs"] - CONFIG["freeze_epochs"],
                eta_min=1e-6,
            )
            print(f"Epoch {epoch}: backbone unfrozen.")

        train_loss = run_train(model, train_loader, optimizer, scaler, focal, lovasz, change_loss)
        validation = run_eval(model, validation_loader, use_tta=False, focal=focal, lovasz=lovasz, change_loss=change_loss)
        scheduler.step()
        print(
            f"Epoch {epoch:03d}/{CONFIG['epochs']} | TrainLoss={train_loss:.4f} | "
            f"ValLoss={validation['loss']:.4f} | OA={validation['oa']:.4f} | "
            f"mIoU={validation['miou']:.4f} | CVA-F1={validation['cva_change_f1']:.4f} | "
            f"{time.time() - started:.1f}s"
        )

        if validation["miou"] > best_miou:
            best_miou = validation["miou"]
            patience = 0
            torch.save(
                {
                    "run_name": RUN_NAME,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "config": CONFIG,
                    "val_metrics": validation,
                },
                checkpoint_path,
            )
            print(f"  Saved best V5 checkpoint: mIoU={best_miou:.4f}")
        else:
            patience += 1
            if patience >= CONFIG["early_stop_patience"]:
                print(f"Early stopping at epoch {epoch}; patience={CONFIG['early_stop_patience']}")
                break
    return best_miou


ckpt_path = os.path.join(OUT_DIR, CKPT_NAME)
best_miou = train_full(model, train_loader, val_loader, ckpt_path)
print(f"Training complete. Best validation mIoU: {best_miou:.4f}")


Epoch 001/160 | TrainLoss=2.0659 | ValLoss=2.0116 | OA=0.6665 | mIoU=0.3487 | CVA-F1=0.2746 | 115.2s
  Saved best V5 checkpoint: mIoU=0.3487
Epoch 002/160 | TrainLoss=1.9607 | ValLoss=1.9566 | OA=0.6481 | mIoU=0.3402 | CVA-F1=0.3073 | 114.9s
Epoch 003/160 | TrainLoss=1.9246 | ValLoss=1.9386 | OA=0.6506 | mIoU=0.3497 | CVA-F1=0.3021 | 115.0s
  Saved best V5 checkpoint: mIoU=0.3497
Epoch 004/160 | TrainLoss=1.9055 | ValLoss=1.9172 | OA=0.6659 | mIoU=0.3581 | CVA-F1=0.2982 | 115.1s
  Saved best V5 checkpoint: mIoU=0.3581
Epoch 005/160 | TrainLoss=1.8871 | ValLoss=1.9109 | OA=0.6705 | mIoU=0.3648 | CVA-F1=0.3001 | 115.3s
  Saved best V5 checkpoint: mIoU=0.3648
Epoch 006/160 | TrainLoss=1.8748 | ValLoss=1.9015 | OA=0.7143 | mIoU=0.3946 | CVA-F1=0.3030 | 114.9s
  Saved best V5 checkpoint: mIoU=0.3946
Epoch 007/160 | TrainLoss=1.8624 | ValLoss=1.8924 | OA=0.6943 | mIoU=0.3818 | CVA-F1=0.3107 | 115.2s
Epoch 008/160 | TrainLoss=1.8523 | ValLoss=1.8881 | OA=0.7198 | mIoU=0.3970 | CVA-F1=0.3130 |

## 8. Final Evaluation with Test-Time Augmentation (TTA)

In [9]:
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"V5 checkpoint was not created: {ckpt_path}")

checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded V5 checkpoint from epoch {checkpoint['epoch']}")

final = run_eval(model, val_loader, use_tta=True)
print("\nFinal Evaluation with Test-Time Augmentation (TTA):")
print(f"  Overall Accuracy         : {final['oa']:.4f}")
print(f"  Mean IoU                 : {final['miou']:.4f}")
print(f"  Mean F1                  : {final['mf1']:.4f}")
print(f"  Cohen's Kappa            : {final['kappa']:.4f}")
print("\nChange Detection Agreement (vs CVA Reference):")
print(f"  Precision                : {final['cva_change_precision']:.4f}")
print(f"  Recall                   : {final['cva_change_recall']:.4f}")
print(f"  F1                       : {final['cva_change_f1']:.4f}")
print(f"  IoU                      : {final['cva_change_iou']:.4f}")

class_names = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]
print("\nPer-class LULC metrics:")
for name, iou, f1 in zip(class_names, final["iou_per_class"], final["f1_per_class"]):
    print(f"  {name:12s} IoU={iou:.4f} F1={f1:.4f}")

Loaded V5 checkpoint from epoch 158

Final Evaluation with Test-Time Augmentation (TTA):
  Overall Accuracy         : 0.8565
  Mean IoU                 : 0.6429
  Mean F1                  : 0.7706
  Cohen's Kappa            : 0.8150

Change Detection Agreement (vs CVA Reference):
  Precision                : 0.2632
  Recall                   : 0.7705
  F1                       : 0.3924
  IoU                      : 0.2441

Per-class LULC metrics:
  Trees        IoU=0.6349 F1=0.7767
  Shrubland    IoU=0.4687 F1=0.6382
  Grassland    IoU=0.4786 F1=0.6473
  Cropland     IoU=0.6585 F1=0.7941
  Built-up     IoU=0.8070 F1=0.8932
  Bare/Sparse  IoU=0.4768 F1=0.6457
  Water        IoU=0.9784 F1=0.9891
  Wetland      IoU=0.6402 F1=0.7806


## 9. Confidence-Threshold Sensitivity Analysis

In [10]:
def make_semantic_change(pred1, pred2, confidence1, confidence2, tau):
    return (
        (pred1 != pred2)
        & (confidence1 >= tau)
        & (confidence2 >= tau)
    ).astype(np.uint8)


model.eval()
sensitivity_rows = []
sample_indices = val_patch_idx[:SENSITIVITY_PATCHES_PER_PAIR]
sample_pairs = [(2017, 2025), (2019, 2024), (2017, 2024)]
cached_predictions = []

print("Confidence-threshold sensitivity on a fixed validation subset:")
for year1, year2 in sample_pairs:
    for patch_index in sample_indices:
        image1 = normalize_bands(to_nine_band(YEAR_ARRAYS[year1][patch_index]))
        image2 = normalize_bands(to_nine_band(YEAR_ARRAYS[year2][patch_index]))
        tensor1 = torch.from_numpy(image1).unsqueeze(0).to(device)
        tensor2 = torch.from_numpy(image2).unsqueeze(0).to(device)
        logits1, _ = tta_predict(model, tensor1, tensor1)
        logits2, _ = tta_predict(model, tensor2, tensor2)
        probabilities1 = F.softmax(logits1, dim=1)
        probabilities2 = F.softmax(logits2, dim=1)
        cached_predictions.append((
            logits1.argmax(1).cpu().numpy()[0],
            logits2.argmax(1).cpu().numpy()[0],
            probabilities1.max(dim=1)[0].cpu().numpy()[0],
            probabilities2.max(dim=1)[0].cpu().numpy()[0],
            cva_masks[(year1, year2, patch_index)].astype(np.uint8),
        ))

for tau in SENSITIVITY_TAUS:
    semantic_values = []
    cva_values = []
    for pred1, pred2, conf1, conf2, cva in cached_predictions:
        semantic_values.append(make_semantic_change(pred1, pred2, conf1, conf2, tau).reshape(-1))
        cva_values.append(cva.reshape(-1))
    semantic_flat = np.concatenate(semantic_values)
    cva_flat = np.concatenate(cva_values)
    cva_eval = binary_metrics(semantic_flat, cva_flat)
    row = {
        "tau": tau,
        "semantic_change_coverage": semantic_flat.mean(),
        "cva_change_coverage": cva_flat.mean(),
        "cva_precision": cva_eval["precision"],
        "cva_recall": cva_eval["recall"],
        "cva_change_iou": cva_eval["iou"],
    }
    sensitivity_rows.append(row)
    print(
        f"  tau={tau:.2f} | semantic coverage={row['semantic_change_coverage']:.3%} | "
        f"CVA precision={row['cva_precision']:.3f} | "
        f"CVA recall={row['cva_recall']:.3f} | CVA change-IoU={row['cva_change_iou']:.3f}"
    )

sensitivity_path = os.path.join(OUT_DIR, "v5_confidence_threshold_sensitivity.csv")
with open(sensitivity_path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=sensitivity_rows[0].keys())
    writer.writeheader()
    writer.writerows(sensitivity_rows)
print(f"Saved sensitivity report: {sensitivity_path}")
print(f"Fixed display threshold: {DISPLAY_CONFIDENCE_TAU:.2f}")


Confidence-threshold sensitivity on a fixed validation subset:
  tau=0.40 | semantic coverage=8.848% | CVA precision=0.280 | CVA recall=0.205 | CVA change-IoU=0.134
  tau=0.50 | semantic coverage=4.605% | CVA precision=0.319 | CVA recall=0.121 | CVA change-IoU=0.096
  tau=0.60 | semantic coverage=2.239% | CVA precision=0.357 | CVA recall=0.066 | CVA change-IoU=0.059
  tau=0.70 | semantic coverage=0.874% | CVA precision=0.405 | CVA recall=0.029 | CVA change-IoU=0.028
  tau=0.80 | semantic coverage=0.142% | CVA precision=0.485 | CVA recall=0.006 | CVA change-IoU=0.006
Saved sensitivity report: /kaggle/working/v5_confidence_threshold_sensitivity.csv
Fixed display threshold: 0.60


## 10. Publication Visualization Dashboard (12-Subplot Figures)

In [11]:
LULC_COLORS = np.array([
    [0, 100, 0], [139, 90, 43], [124, 252, 0], [255, 215, 0],
    [255, 0, 0], [210, 180, 140], [0, 0, 255], [0, 128, 128],
], dtype=np.float32) / 255.0

TRANSITION_COLORS = {
    "No Change": [220 / 255, 220 / 255, 220 / 255],
    "Urban Expansion (Veg→Built)": [1.0, 0.0, 0.0],
    "Land Clearing (Veg→Bare)": [148 / 255, 0.0, 211 / 255],
    "Water Drying (Water→Bare/Built)": [1.0, 140 / 255, 0.0],
    "Construction (Bare→Built)": [139 / 255, 0.0, 0.0],
    "Vegetation Regrowth": [0.0, 1.0, 127 / 255],
    "Other Change": [0.0, 191 / 255, 1.0],
}

AGREEMENT_COLORS = {
    "Both Indicate Change": [0.0, 0.75, 0.0],
    "Both Indicate No Change": [0.85, 0.85, 0.85],
    "CVA-Only Change": [0.9, 0.0, 0.0],
    "Semantic-Only Change": [1.0, 0.55, 0.0],
}

CLASS_NAMES = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]


def label_to_rgb(label_map, colors=LULC_COLORS):
    rgb = np.zeros((*label_map.shape, 3), dtype=np.float32)
    for class_index in range(len(colors)):
        rgb[label_map == class_index] = colors[class_index]
    return rgb


def make_shared_rgb_pair(image1, image2, lower_percentile=2, upper_percentile=98):
    rgb1 = np.stack([image1[2], image1[1], image1[0]], axis=-1)
    rgb2 = np.stack([image2[2], image2[1], image2[0]], axis=-1)
    joint_pixels = np.concatenate([rgb1.reshape(-1, 3), rgb2.reshape(-1, 3)], axis=0)
    low = np.percentile(joint_pixels, lower_percentile, axis=0)
    high = np.percentile(joint_pixels, upper_percentile, axis=0)
    scale = np.maximum(high - low, 1e-6)

    def stretch(rgb):
        return np.clip((rgb - low) / scale, 0, 1)

    return stretch(rgb1), stretch(rgb2)


def compute_from_to_map(pred1, pred2, semantic_change):
    rgb = np.zeros((*pred1.shape, 3), dtype=np.float32)
    rgb[:] = TRANSITION_COLORS["No Change"]
    vegetation = {0, 1, 2, 3}
    changed = semantic_change.astype(bool)
    urban = changed & np.isin(pred1, list(vegetation)) & (pred2 == 4)
    clearing = changed & np.isin(pred1, list(vegetation)) & (pred2 == 5)
    drying = changed & np.isin(pred1, [6, 7]) & np.isin(pred2, [4, 5])
    construction = changed & (pred1 == 5) & (pred2 == 4)
    regrowth = changed & np.isin(pred1, [4, 5]) & np.isin(pred2, list(vegetation))
    assigned = urban | clearing | drying | construction | regrowth
    rgb[urban] = TRANSITION_COLORS["Urban Expansion (Veg→Built)"]
    rgb[clearing] = TRANSITION_COLORS["Land Clearing (Veg→Bare)"]
    rgb[drying] = TRANSITION_COLORS["Water Drying (Water→Bare/Built)"]
    rgb[construction] = TRANSITION_COLORS["Construction (Bare→Built)"]
    rgb[regrowth] = TRANSITION_COLORS["Vegetation Regrowth"]
    rgb[changed & ~assigned] = TRANSITION_COLORS["Other Change"]
    return rgb


def compute_agreement_map(cva_binary, semantic_change):
    rgb = np.zeros((*cva_binary.shape, 3), dtype=np.float32)
    rgb[:] = AGREEMENT_COLORS["Both Indicate No Change"]
    both_change = (cva_binary == 1) & (semantic_change == 1)
    cva_only = (cva_binary == 1) & (semantic_change == 0)
    semantic_only = (cva_binary == 0) & (semantic_change == 1)
    rgb[both_change] = AGREEMENT_COLORS["Both Indicate Change"]
    rgb[cva_only] = AGREEMENT_COLORS["CVA-Only Change"]
    rgb[semantic_only] = AGREEMENT_COLORS["Semantic-Only Change"]
    return rgb


samples = [
    (2017, 2025, val_patch_idx[10 % len(val_patch_idx)]),
    (2019, 2024, val_patch_idx[35 % len(val_patch_idx)]),
    (2017, 2024, val_patch_idx[75 % len(val_patch_idx)]),
    (2019, 2025, val_patch_idx[120 % len(val_patch_idx)]),
    (2017, 2019, val_patch_idx[180 % len(val_patch_idx)]),
    (2024, 2025, val_patch_idx[240 % len(val_patch_idx)]),
    (2017, 2025, val_patch_idx[310 % len(val_patch_idx)]),
    (2019, 2024, val_patch_idx[390 % len(val_patch_idx)]),
    (2017, 2024, val_patch_idx[480 % len(val_patch_idx)]),
    (2019, 2025, val_patch_idx[550 % len(val_patch_idx)]),
]
main_report_sample_numbers = {3, 5, 10}
summary_rows = []

model.eval()
print(f"Generating {len(samples)} V5 figures with TTA...")
for sample_number, (year1, year2, patch_index) in enumerate(samples, start=1):
    image1 = normalize_bands(to_nine_band(YEAR_ARRAYS[year1][patch_index]))
    image2 = normalize_bands(to_nine_band(YEAR_ARRAYS[year2][patch_index]))
    gt1 = LABEL_FOR_YEAR[year1][patch_index]
    gt2 = LABEL_FOR_YEAR[year2][patch_index]
    cva_binary = cva_masks[(year1, year2, patch_index)].astype(np.uint8)
    rgb1, rgb2 = make_shared_rgb_pair(image1, image2)

    tensor1 = torch.from_numpy(image1).unsqueeze(0).to(device)
    tensor2 = torch.from_numpy(image2).unsqueeze(0).to(device)
    logits1, _ = tta_predict(model, tensor1, tensor1)
    logits2, _ = tta_predict(model, tensor2, tensor2)
    probabilities1 = F.softmax(logits1, dim=1)
    probabilities2 = F.softmax(logits2, dim=1)
    prediction1 = logits1.argmax(1).cpu().numpy()[0]
    prediction2 = logits2.argmax(1).cpu().numpy()[0]
    confidence1 = probabilities1.max(dim=1)[0].cpu().numpy()[0]
    confidence2 = probabilities2.max(dim=1)[0].cpu().numpy()[0]
    semantic_change = make_semantic_change(
        prediction1, prediction2, confidence1, confidence2, DISPLAY_CONFIDENCE_TAU
    )
    agreement = compute_agreement_map(cva_binary, semantic_change)
    transition = compute_from_to_map(prediction1, prediction2, semantic_change)
    change_metrics = binary_metrics(semantic_change.reshape(-1), cva_binary.reshape(-1))

    correct1 = prediction1 == gt1
    correct2 = prediction2 == gt2
    error1 = np.zeros((*gt1.shape, 3), dtype=np.float32)
    error2 = np.zeros((*gt2.shape, 3), dtype=np.float32)
    error1[correct1], error1[~correct1] = [0.0, 0.8, 0.0], [0.9, 0.0, 0.0]
    error2[correct2], error2[~correct2] = [0.0, 0.8, 0.0], [0.9, 0.0, 0.0]

    figure, axes = plt.subplots(3, 4, figsize=(24, 18))
    panels = [
        (axes[0, 0], rgb1, f"T1 ({year1}) Sentinel-2 RGB", None),
        (axes[0, 1], label_to_rgb(gt1), f"Ground Truth ({year1})", None),
        (axes[0, 2], label_to_rgb(prediction1), f"Predicted LULC ({year1}) — TTA", None),
        (axes[0, 3], error1, f"LULC Error Map T1 ({correct1.mean() * 100:.1f}%)", None),
        (axes[1, 0], rgb2, f"T2 ({year2}) Sentinel-2 RGB", None),
        (axes[1, 1], label_to_rgb(gt2), f"Ground Truth ({year2})", None),
        (axes[1, 2], label_to_rgb(prediction2), f"Predicted LULC ({year2}) — TTA", None),
        (axes[1, 3], error2, f"LULC Error Map T2 ({correct2.mean() * 100:.1f}%)", None),
        (axes[2, 0], cva_binary, f"Spectral Change / CVA ({cva_binary.mean() * 100:.1f}%)", "gray"),
        (axes[2, 1], semantic_change, f"Semantic Change (confidence ≥ {DISPLAY_CONFIDENCE_TAU:.2f}): {semantic_change.mean() * 100:.1f}%", "gray"),
        (axes[2, 2], agreement, f"CVA vs Semantic Signals (change IoU={change_metrics['iou']:.3f})", None),
        (axes[2, 3], transition, "Confidence-Filtered Predicted Transitions", None),
    ]
    for axis, image, title, cmap in panels:
        axis.imshow(image, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
        axis.set_title(title, fontweight="bold", fontsize=11)
        axis.axis("off")

    built1 = (prediction1 == 4).mean() * 100
    built2 = (prediction2 == 4).mean() * 100
    stats_text = (
        f"T1 LULC accuracy: {correct1.mean() * 100:.1f}%   |   "
        f"T2 LULC accuracy: {correct2.mean() * 100:.1f}%   |   "
        f"CVA coverage: {cva_binary.mean() * 100:.1f}%   |   "
        f"Semantic coverage: {semantic_change.mean() * 100:.1f}%   |   "
        f"CVA–semantic change IoU: {change_metrics['iou']:.3f}   |   "
        f"Built-up: {built1:.1f}% → {built2:.1f}%"
    )
    figure.text(0.5, 0.025, stats_text, ha="center", va="bottom", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="gray", alpha=0.95))
    figure.suptitle(
        f"Multi-Temporal LULC Analysis | Sample {sample_number}/10 | Patch {patch_index} | {year1} → {year2}",
        fontsize=17, fontweight="bold", y=0.985,
    )
    figure.tight_layout(rect=[0.0, 0.07, 1.0, 0.96])
    subset = "main" if sample_number in main_report_sample_numbers else "appendix"
    figure_path = os.path.join(
        OUT_DIR, f"{subset}_v5_sample_{sample_number}_patch_{patch_index}_{year1}_{year2}.png"
    )
    figure.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.close(figure)
    print(f"  [{sample_number}/10] Saved {subset} figure: {figure_path}")

    row = {
        "sample": sample_number,
        "subset": subset,
        "patch": patch_index,
        "year_t1": year1,
        "year_t2": year2,
        "t1_accuracy": correct1.mean(),
        "t2_accuracy": correct2.mean(),
        "cva_change_coverage": cva_binary.mean(),
        "semantic_change_coverage": semantic_change.mean(),
        "cva_semantic_change_iou": change_metrics["iou"],
    }
    for class_index, class_name in enumerate(CLASS_NAMES):
        key = class_name.lower().replace("/", "_").replace(" ", "_")
        row[f"t1_{key}_pct"] = (prediction1 == class_index).mean() * 100
        row[f"t2_{key}_pct"] = (prediction2 == class_index).mean() * 100
    summary_rows.append(row)

summary_path = os.path.join(OUT_DIR, "v5_sample_summary.csv")
with open(summary_path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=summary_rows[0].keys())
    writer.writeheader()
    writer.writerows(summary_rows)
print(f"Saved sample summary: {summary_path}")

showcase_rows = [row for row in summary_rows if row["subset"] == "main"]
figure, axes = plt.subplots(len(showcase_rows), 1, figsize=(12, 3.8 * len(showcase_rows)), sharex=True)
if len(showcase_rows) == 1:
    axes = [axes]
y = np.arange(NUM_CLASSES)
for axis, row in zip(axes, showcase_rows):
    t1_values = [row[f"t1_{name.lower().replace('/', '_').replace(' ', '_')}_pct"] for name in CLASS_NAMES]
    t2_values = [row[f"t2_{name.lower().replace('/', '_').replace(' ', '_')}_pct"] for name in CLASS_NAMES]
    axis.barh(y - 0.18, t1_values, height=0.35, label=f"T1 ({row['year_t1']})")
    axis.barh(y + 0.18, t2_values, height=0.35, label=f"T2 ({row['year_t2']})")
    axis.set_yticks(y, labels=CLASS_NAMES)
    axis.set_title(f"Class-area comparison — main sample {row['sample']} (patch {row['patch']})", fontweight="bold")
    axis.set_xlabel("Predicted area (%)")
    axis.grid(axis="x", alpha=0.25)
    axis.legend()
figure.tight_layout()
area_chart_path = os.path.join(OUT_DIR, "v5_main_sample_class_area_comparison.png")
figure.savefig(area_chart_path, dpi=180, bbox_inches="tight")
plt.close(figure)
print(f"Saved class-area comparison: {area_chart_path}")


Generating 10 V5 figures with TTA...
  [1/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_1_patch_1958_2017_2025.png
  [2/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_2_patch_1680_2019_2024.png
  [3/10] Saved main figure: /kaggle/working/main_v5_sample_3_patch_850_2017_2024.png
  [4/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_4_patch_417_2019_2025.png
  [5/10] Saved main figure: /kaggle/working/main_v5_sample_5_patch_1008_2017_2019.png
  [6/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_6_patch_1077_2024_2025.png
  [7/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_7_patch_2544_2017_2025.png
  [8/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_8_patch_3108_2019_2024.png
  [9/10] Saved appendix figure: /kaggle/working/appendix_v5_sample_9_patch_1257_2017_2024.png
  [10/10] Saved main figure: /kaggle/working/main_v5_sample_10_patch_2731_2019_2025.png
Saved sample summary: /kaggle/working/v5_sample

## 11. Class Legends & Submission Asset Export

In [12]:
lulc_legend, lulc_axis = plt.subplots(figsize=(4.5, 3.2))
lulc_axis.axis("off")
lulc_handles = [mpatches.Patch(color=LULC_COLORS[index], label=name) for index, name in enumerate(CLASS_NAMES)]
lulc_axis.legend(handles=lulc_handles, loc="center", fontsize=11, frameon=True, title="LULC Classes", title_fontsize=12)
lulc_legend.tight_layout()
lulc_legend_path = os.path.join(OUT_DIR, "v5_lulc_class_legend.png")
lulc_legend.savefig(lulc_legend_path, dpi=180, bbox_inches="tight")
plt.close(lulc_legend)

change_legend, change_axis = plt.subplots(figsize=(10, 4.5))
change_axis.axis("off")
transition_handles = [mpatches.Patch(color=color, label=name) for name, color in TRANSITION_COLORS.items()]
agreement_handles = [mpatches.Patch(color=color, label=name) for name, color in AGREEMENT_COLORS.items()]
change_axis.legend(
    handles=transition_handles + agreement_handles,
    loc="center", fontsize=10, frameon=True, ncol=2,
    title="Transition categories and CVA–semantic signal comparison", title_fontsize=12,
)
change_legend.tight_layout()
change_legend_path = os.path.join(OUT_DIR, "v5_change_analysis_legend.png")
change_legend.savefig(change_legend_path, dpi=180, bbox_inches="tight")
plt.close(change_legend)

print(f"Saved: {lulc_legend_path}")
print(f"Saved: {change_legend_path}")
print("\nFinal-report wording:")
print("- Reference LULC metrics are evaluated against ESA WorldCover 10m ground truth.")
print("- CVA provides physical spectral change detection; semantic change maps categorical from-to transitions.")
print("- Confidence-filtered semantic change is derived from T1/T2 LULC predictions.")
print("- Manual reference-point validation remains the recommended next evaluation step.")


Saved: /kaggle/working/v5_lulc_class_legend.png
Saved: /kaggle/working/v5_change_analysis_legend.png

Final-report wording:
- Reference LULC metrics are evaluated against ESA WorldCover 10m ground truth.
- CVA provides physical spectral change detection; semantic change maps categorical from-to transitions.
- Confidence-filtered semantic change is derived from T1/T2 LULC predictions.
- Manual reference-point validation remains the recommended next evaluation step.
